In [0]:
from pyspark.sql import functions as F

# reading data
silver_df = spark.readStream.table("crp_arb_dev.silver.crypto_data_15m")

gold_15m_aggregated = silver_df \
    .withWatermark("event_timestamp", "15 minutes") \
    .groupBy(
        F.window("event_timestamp", "15 minutes"),
        "ticker"
    ).agg(
        F.first("open").alias("open"),
        F.last("close").alias("close"),
        F.max("high_price").alias("high"),
        F.min("low_price").alias("low"),
        F.sum("trading_volume").alias("volume")
    )


gold_df = gold_15m_aggregated \
    # cols for table
    .select(
        F.col("window.start").alias("timestamp"),
        "ticker", "open", "close", "high", "low", "voluem"
    ) \
    .withColumn("price_diff_pct", F.round(((F.col("close") - F.col("open")) / F.col("open")) * 100, 2)) \
    .withColumn("volatility_pct", F.round(((F.col("high") - F.col("low")) / F.col("low")) * 100, 2)) \
    .withColumn("market_signal", 
        F.when(F.col("price_diff_pct") > 0.5, "BULLISH")
         .when(F.col("price_diff_pct") < -0.5, "BEARISH")
         .otherwise("STABLE")) \
    .withColumn("risk_level", 
        F.when(F.col("volatility_pct") > 1.2, "HIGH RISK")
         .when(F.col("volatility_pct") > 0.6, "MEDIUM")
         .otherwise("LOW"))
    
container_path = "abfss://crp-arb-cnt@sadlsdev.dfs.core.windows.net"
query_15m = gold_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{container_path}/_checkpoints/gold_15m_window") \
    .option("path", f"{container_path}/gold/crypto_data_15m") \
    .trigger(availableNow=True) \
    .toTable("crp_arb_dev.gold.crypto_data_15m_window")
    
query_15m.awaitTermination()